# Week 1: Strategic Planning and Data Exploration in Logistics
### E-Commerce Order Fulfillment Analytics — Strategic Report

**Dataset:** E-Commerce Order Fulfillment Dataset (50,000 records) — Kaggle
**Objective:** Explore the dataset, validate its quality, and compute baseline logistics KPIs to support the Week 1 strategic plan.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

## 2. Load Dataset

In [2]:
df = pd.read_csv("E-Commerce_Order_Fulfillment_Dataset.csv")

## 3. Dataset Overview

In [3]:
# Check dataset shape
df.shape

(50000, 10)

In [4]:
# Display first records
df.head()

,Order_ID,Customer_Region,Product_Category,Order_Date,Ship_Date,Delivery_Date,Shipping_Mode,Shipping_Cost,Delivery_Status,Delivery_Days
0,OR10000,West,Home,5/12/2024,5/16/2024,5/26/2024,Standard,122,Delivered,10
1,OR10001,Central,Home,4/19/2025,4/24/2025,4/26/2025,Standard,143,Delivered,2
2,OR10002,East,Home,7/2/2025,7/4/2025,7/11/2025,Standard,107,Delayed,7
3,OR10003,Central,Grocery,10/4/2025,10/8/2025,10/16/2025,Standard,145,Delayed,8
4,OR10004,Central,Sports,10/31/2023,11/2/2023,11/7/2023,Standard,121,Delivered,5


In [5]:
# Check dataset information
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Order_ID          50000 non-null  str  
 1   Customer_Region   50000 non-null  str  
 2   Product_Category  50000 non-null  str  
 3   Order_Date        50000 non-null  str  
 4   Ship_Date         50000 non-null  str  
 5   Delivery_Date     50000 non-null  str  
 6   Shipping_Mode     50000 non-null  str  
 7   Shipping_Cost     50000 non-null  int64
 8   Delivery_Status   50000 non-null  str  
 9   Delivery_Days     50000 non-null  int64
dtypes: int64(2), str(8)
memory usage: 3.8 MB


In [6]:
# Descriptive statistics — numeric columns
df.describe()

,Shipping_Cost,Delivery_Days
count,50000.000000,50000.000000
mean,138.838280,5.989760
std,72.498689,2.573649
min,50.000000,2.000000
25%,88.000000,4.000000
50%,121.000000,6.000000
75%,164.000000,8.000000
max,399.000000,10.000000


In [7]:
# Descriptive statistics — categorical columns
df.describe(include='object')

/tmp/ipykernel_111/2031566831.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include=['object', 'str'])


,Order_ID,Customer_Region,Product_Category,Order_Date,Ship_Date,Delivery_Date,Shipping_Mode,Delivery_Status
count,50000,50000,50000,50000,50000,50000,50000,50000
unique,50000,5,6,1461,1466,1473,3,3
top,OR10000,Central,Sports,7/27/2024,3/19/2022,10/30/2022,Standard,Delivered
freq,1,10117,8380,52,54,57,29973,40062


## 4. Data Quality Checks

Before computing any KPIs, the dataset is validated for missing values and duplicate records.

In [8]:
# Check missing values
missing_values = df.isnull().sum()
missing_values

Order_ID            0
Customer_Region     0
Product_Category    0
Order_Date          0
Ship_Date           0
Delivery_Date       0
Shipping_Mode       0
Shipping_Cost       0
Delivery_Status     0
Delivery_Days       0
dtype: int64

In [9]:
# Missing value percentage
missing_percentage = missing_values / len(df) * 100
missing_percentage

Order_ID            0.0
Customer_Region     0.0
Product_Category    0.0
Order_Date          0.0
Ship_Date           0.0
Delivery_Date       0.0
Shipping_Mode       0.0
Shipping_Cost       0.0
Delivery_Status     0.0
Delivery_Days       0.0
dtype: float64

In [10]:
# Check for duplicate rows
df.duplicated().sum()

np.int64(0)

**Result:** The dataset contains no missing values and no duplicate rows across all 50,000 records.

## 5. Explore Categorical Fields

In [11]:
# Unique customer regions
Customer_Regions = df['Customer_Region'].unique()
Customer_Regions

<StringArray>
['West', 'Central', 'East', 'South', 'North']
Length: 5, dtype: str

In [12]:
# Unique product categories
Product_categories = df['Product_Category'].unique()
print(Product_categories)

<StringArray>
['Home', 'Grocery', 'Sports', 'Fashion', 'Beauty', 'Electronics']
Length: 6, dtype: str


In [13]:
# Unique shipping modes
Shipping_modes = df['Shipping_Mode'].unique()
print(Shipping_modes)

<StringArray>
['Standard', 'Express', 'Same Day']
Length: 3, dtype: str


In [14]:
# Unique delivery statuses
Delivery_status = df['Delivery_Status'].unique()
print(Delivery_status)

<StringArray>
['Delivered', 'Delayed', 'Returned']
Length: 3, dtype: str


In [15]:
# Delivery status counts
delivery_status_counts = df['Delivery_Status'].value_counts()
delivery_status_counts

Delivery_Status
Delivered    40062
Delayed       7505
Returned      2433
Name: count, dtype: int64

In [16]:
# Delivery status percentage
delivery_status_percentage = df['Delivery_Status'].value_counts(normalize=True) * 100
delivery_status_percentage

Delivery_Status
Delivered    80.124
Delayed      15.010
Returned      4.866
Name: proportion, dtype: float64

## 6. Derive Cycle-Time Fields

`Delivery_Days` (as provided) measures ship-to-delivery transit time only. Two additional fields are derived to capture the full order lifecycle:
- **Order_to_Ship_Days** — warehouse processing time
- **Order_to_Delivery_Days** — total end-to-end fulfillment cycle time

In [17]:
date_columns = ['Order_Date', 'Ship_Date', 'Delivery_Date']

for col in date_columns:
    df[col] = pd.to_datetime(df[col])

df['Order_to_Ship_Days'] = (df['Ship_Date'] - df['Order_Date']).dt.days
df['Order_to_Delivery_Days'] = (df['Delivery_Date'] - df['Order_Date']).dt.days

# Check dataset shape after adding new columns
df.shape

(50000, 12)

## 7. Key Performance Indicators (KPIs)

Seven KPIs are computed to establish the baseline for order fulfillment performance.

In [18]:
# KPI 1: Order Fulfillment Cycle Time
avg_cycle_time = df['Order_to_Delivery_Days'].mean()
print("Order Fulfillment Cycle Time:", round(avg_cycle_time, 2), "days")

Order Fulfillment Cycle Time: 8.48 days


In [19]:
# KPI 2: Ship Processing Time
avg_ship_time = df['Order_to_Ship_Days'].mean()
print("Ship Processing Time:", round(avg_ship_time, 2), "days")

Ship Processing Time: 2.49 days


In [20]:
# KPI 3: Delivery Delay Rate
delay_count = (df['Delivery_Status'] == 'Delayed').sum()
delay_rate = (df['Delivery_Status'] == 'Delayed').mean() * 100
print("Delayed Orders:", delay_count)
print("Delivery Delay Rate:", round(delay_rate, 2), "%")

Delayed Orders: 7505
Delivery Delay Rate: 15.01 %


In [21]:
# KPI 4: Return Rate
return_rate = (df['Delivery_Status'] == 'Returned').mean() * 100
print("Return Rate:", round(return_rate, 2), "%")

Return Rate: 4.87 %


In [22]:
# KPI 5: Delivery Success Rate
delivery_success_rate = (df['Delivery_Status'] == 'Delivered').mean() * 100
print("Delivery Success Rate:", round(delivery_success_rate, 2), "%")

Delivery Success Rate: 80.12 %


In [23]:
# KPI 6: Average Shipping Cost
average_shipping_cost = df['Shipping_Cost'].mean()
print("Average Shipping Cost:", round(average_shipping_cost, 2))

Average Shipping Cost: 138.84


In [24]:
# KPI 7: Shipping Cost by Mode
cost_by_mode = df.groupby('Shipping_Mode')['Shipping_Cost'].mean()
cost_by_mode

Shipping_Mode
Express     164.581082
Same Day    299.587263
Standard     99.287626
Name: Shipping_Cost, dtype: float64

## 8. KPI Summary Table

| # | KPI | Result |
|---|---|---|
| 1 | Order Fulfillment Cycle Time | 8.48 days |
| 2 | Ship Processing Time | 2.49 days |
| 3 | Delivery Delay Rate | 15.01% (7,505 orders) |
| 4 | Return Rate | 4.87% (2,433 orders) |
| 5 | Delivery Success Rate | 80.12% (40,062 orders) |
| 6 | Average Shipping Cost | $138.84 |
| 7 | Shipping Cost by Mode | Standard $99.29 / Express $164.58 / Same Day $299.59 |

## 9. Summary of Findings

- The dataset is complete and clean: no missing values, no duplicate records, across all 50,000 orders.
- `Delivery_Days` measures ship-to-delivery transit time only (2.49-day processing time is separate and must be added for true end-to-end cycle time of 8.48 days).
- The overall delay rate stands at 15.01%, with a return rate of 4.87% and a delivery success rate of 80.12%.
- Shipping cost varies substantially by mode: Same Day shipping costs roughly 3x Standard shipping on average.

## 10. Next Steps

This notebook establishes the Week 1 baseline. Subsequent weeks will:
- **Week 2:** Perform a deeper data-cleaning and preprocessing pipeline.
- **Week 3:** Conduct exploratory data analysis with visualizations across region, category, and shipping mode.
- **Week 4:** Build and evaluate regression and classification models to predict delivery time and delay risk.